# Accessing Web Archive Data

<img src="https://ndha-public-data-ap-southeast-2.s3.ap-southeast-2.amazonaws.com/iPRES-2025/resources/nlnz-webarchive-public-portal.png" alt="NLNZ Selective Web Archive portal" border="0">

## Overview

This notebook demonstrates how to access and query web archive data from several open web archives. It provides a foundation for working with web archives using various protocols and APIs.

### Learning Objectives

1. Query web archive data using the Memento protocol
2. Retrieve and interpret capture histories using TimeMaps
3. Access archived content using the CDX API
4. Identify archived content types for analysis

This notebook serves as an introduction to web archive access methods that will be built upon in subsequent notebooks.

## Environment Setup

### Installing Required Python Packages

The following packages are necessary for working with web archives:

In [1]:
# Install core dependencies for web archive processing
!pip -q install warcio validators boto3 s3fs bs4

# Install packages for webpage screenshots (optional visualization)
!pip -q install selenium chromedriver-autoinstaller


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Install the NLNZ Web Archive Toolkit
!pip -q install -i https://test.pypi.org/simple/ wa-nlnz-toolkit==0.3.3
# !pip -q install -e ../../


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
# Import the NLNZ Web Archive Toolkit
import wa_nlnz_toolkit as want
import pandas as pd
import datetime
from bs4 import BeautifulSoup

### Configuring Web Archive Endpoints

By default this notebook is configured to query the New Zealand Web Archive. To override this configuration, set the following variables:

In [4]:
# New Zealand Web Archive
want.set_memento_url("https://ndhadeliver.natlib.govt.nz/webarchive")

# Norwegian Web Archive
# want.set_memento_url("https://nettarkivet.nb.no/search")

# Australian Web Archive
# want.set_memento_url("https://web.archive.org.au/awa")

# Arquivo.pt
# want.set_memento_url("https://arquivo.pt/wayback")


## 1. Querying Web Archives with the Memento Protocol

### Introduction to Memento

![image](https://ndha-public-data-ap-southeast-2.s3.ap-southeast-2.amazonaws.com/iPRES-2025/resources/memento.png)

The **Memento protocol** provides a standardized way to access archived versions of web pages across different web archives. It offers machine-readable information about web captures and simplifies the process of finding historical versions of web content.

### Key Memento Components

The NLNZ web archive supports three main Memento features:

1. **TimeGate** - Retrieves the version of a page closest to a specified date
2. **TimeMap** - Provides a list of all archived versions of a page
3. **Memento** - Represents a specific archived version with options to control presentation

Let's explore how to use these features with the NLNZ Web Archive Toolkit.

### Basic Memento Queries

We'll start by querying the latest capture of a website using the Memento protocol:

In [5]:
# Define target website

# New Zealand example
webpage = "https://natlib.govt.nz/"

# Norwegian example
# webpage = "http://www.met.no"

# Australian example
# webpage = "https://cobb.qm.qld.gov.au/"

# Portuguese example
# webpage = "https://www.bbc.co.uk"
# webpage = "https://www.abola.pt"


# Query the latest capture using Memento
# This returns the raw response headers
dict(want.query_memento(webpage).headers)

{'Date': 'Fri, 10 Apr 2026 00:52:25 GMT',
 'Server': 'Apache/2.4.6 (Red Hat Enterprise Linux) OpenSSL/1.0.2k-fips mod_fcgid/2.3.9',
 'Content-Type': 'text/html; charset=UTF-8',
 'Content-Length': '6090',
 'Link': '<https://natlib.govt.nz/>; rel="original", <https://ndhadeliver.natlib.govt.nz/webarchive/https://natlib.govt.nz/>; rel="timegate", <https://ndhadeliver.natlib.govt.nz/webarchive/timemap/link/https://natlib.govt.nz/>; rel="timemap"; type="application/link-format", <https://ndhadeliver.natlib.govt.nz/webarchive/20260308130033mp_/https://natlib.govt.nz/>; rel="memento"; datetime="Sun, 08 Mar 2026 13:00:33 GMT"',
 'Vary': 'accept-datetime',
 'Expect-CT': 'max-age=86400, enforce',
 'X-XSS-Protection': '1; mode=block',
 'X-Content-Type-Options': 'nosniff',
 'X-Permitted-Cross-Domain-Policies': 'none',
 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains',
 'Referrer-Policy': 'no-referrer',
 'Keep-Alive': 'timeout=5, max=100',
 'Connection': 'Keep-Alive'}

In [6]:
# Get a more structured representation of Memento URLs
want.get_memento_urls(webpage)

{'original': 'https://natlib.govt.nz/',
 'timegate': 'https://ndhadeliver.natlib.govt.nz/webarchive/https://natlib.govt.nz/',
 'timemap': 'https://ndhadeliver.natlib.govt.nz/webarchive/timemap/link/https://natlib.govt.nz/',
 'memento': 'https://ndhadeliver.natlib.govt.nz/webarchive/20260308130033mp_/https://natlib.govt.nz/'}

### Understanding Memento Link Types

The Memento response contains several important link types:

- **original**: The original URL that was archived (e.g., https://covid19.govt.nz/)
- **timegate**: The URL used to request archived versions (e.g., https://ndhadeliver.natlib.govt.nz/webarchive/https://covid19.govt.nz/)
- **timemap**: URL that lists all available captures (e.g., https://ndhadeliver.natlib.govt.nz/webarchive/timemap/link/https://covid19.govt.nz/)
- **memento**: URL of the specific archived version (e.g., https://ndhadeliver.natlib.govt.nz/webarchive/20250728214105mp_/https://covid19.govt.nz/)

By default, the *memento* link points to the latest capture. We can also request a capture closest to a specific date:

In [7]:
# Query for a capture closest to January 1, 2020
dt_required = datetime.datetime(2020, 1, 1, 0, 0, 0)
dict(want.query_memento(webpage, dt=dt_required).headers)

{'Date': 'Fri, 10 Apr 2026 00:52:33 GMT',
 'Server': 'Apache/2.4.6 (Red Hat Enterprise Linux) OpenSSL/1.0.2k-fips mod_fcgid/2.3.9',
 'Content-Type': 'text/html; charset=UTF-8',
 'Content-Length': '6146',
 'Link': '<https://natlib.govt.nz/>; rel="original", <https://ndhadeliver.natlib.govt.nz/webarchive/https://natlib.govt.nz/>; rel="timegate", <https://ndhadeliver.natlib.govt.nz/webarchive/timemap/link/https://natlib.govt.nz/>; rel="timemap"; type="application/link-format", <https://ndhadeliver.natlib.govt.nz/webarchive/20200130060111mp_/https://natlib.govt.nz/>; rel="memento"; datetime="Thu, 30 Jan 2020 06:01:11 GMT"',
 'Vary': 'accept-datetime',
 'X-XSS-Protection': '1; mode=block',
 'X-Content-Type-Options': 'nosniff',
 'X-Permitted-Cross-Domain-Policies': 'none',
 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains',
 'Referrer-Policy': 'no-referrer',
 'Expect-CT': 'max-age=86400, enforce',
 'Keep-Alive': 'timeout=5, max=100',
 'Connection': 'Keep-Alive'}

In [8]:
# Get structured Memento URLs for a specific date
want.get_memento_urls(webpage, dt=dt_required)

{'original': 'https://natlib.govt.nz/',
 'timegate': 'https://ndhadeliver.natlib.govt.nz/webarchive/https://natlib.govt.nz/',
 'timemap': 'https://ndhadeliver.natlib.govt.nz/webarchive/timemap/link/https://natlib.govt.nz/',
 'memento': 'https://ndhadeliver.natlib.govt.nz/webarchive/20200130060111mp_/https://natlib.govt.nz/'}

In [10]:
# Query another website to see its Memento links
want.query_memento("www.niwa.co.nz").links

{'original': {'url': 'http://www.niwa.co.nz/', 'rel': 'original'},
 'timegate': {'url': 'https://ndhadeliver.natlib.govt.nz/webarchive/http://www.niwa.co.nz/',
  'rel': 'timegate'},
 'timemap': {'url': 'https://ndhadeliver.natlib.govt.nz/webarchive/timemap/link/http://www.niwa.co.nz/',
  'rel': 'timemap',
  'type': 'application/link-format'},
 'memento': {'url': 'https://ndhadeliver.natlib.govt.nz/webarchive/20200318073807mp_/http://www.niwa.co.nz/',
  'rel': 'memento',
  'datetime': 'Wed, 18 Mar 2020 07:38:07 GMT'}}

>💡 <strong>Try it:</strong> Copy the memento URL from the output and paste it into a web browser to see the URL load from the Web Archive.

### Retrieving Complete Capture History with TimeMap

The Memento TimeMap provides a comprehensive list of all captures for a given webpage. The NLNZ web archive supports multiple TimeMap formats (link, cdxj, and json).

Let's retrieve the TimeMap for our example website:

In [ ]:
# Get the TimeMap for the National Library website

# New Zealand Web Archive
webpage = "www.natlib.govt.nz"

# Norwegian Web Archive
# webpage = "www.met.no"

# Australian Web Archive
# webpage = "cobb.qm.qld.gov.au/"

# Arquivo.pt
# webpage = "https://www.abola.pt"

want.get_timemap(webpage)

https://ndhadeliver.natlib.govt.nz/webarchive/timemap/json/https://natlib.govt.nz/


,urlkey,url,mime,status,digest,redirect,robotflags,length,offset,filename,source,source-coll,access_url
timestamp,,,,,,,,,,,,,
2020-01-30 06:01:06,"nz,govt,natlib)/",http://natlib.govt.nz/,text/html,301,Q2NZOEQVJ4XX7ZDMCJH63Q4ZJZ3NX66N,-,-,0,151529,V1-FL50765403.warc,webarchive,webarchive,https://ndhadeliver.natlib.govt.nz/webarchive/20200130060106/http://natlib.govt.nz/
2020-01-30 06:01:11,"nz,govt,natlib)/",https://natlib.govt.nz/,text/html,200,DHRWFIKXJPMQV3BNEWYTURZ5ZJ4PXSDT,-,-,0,162188,V1-FL50765403.warc,webarchive,webarchive,https://ndhadeliver.natlib.govt.nz/webarchive/20200130060111/https://natlib.govt.nz/
2020-05-30 06:35:23,"nz,govt,natlib)/",https://natlib.govt.nz/,text/html,200,J5DYVJWHJ7MKZDKQCH2B6CVY6YJOF5IY,-,-,0,3477,V1-FL57988342.warc,webarchive,webarchive,https://ndhadeliver.natlib.govt.nz/webarchive/20200530063523/https://natlib.govt.nz/
2020-05-30 23:42:00,"nz,govt,natlib)/",http://natlib.govt.nz/,text/html,301,Q2NZOEQVJ4XX7ZDMCJH63Q4ZJZ3NX66N,-,-,0,76339245,V1-FL57988457.warc,webarchive,webarchive,https://ndhadeliver.natlib.govt.nz/webarchive/20200530234200/http://natlib.govt.nz/
2020-07-30 07:00:58,"nz,govt,natlib)/",https://natlib.govt.nz/,text/html,200,24OAFLGTCDDIOQZ6PINZAXBPJGAV73WW,-,-,0,27225,V1-FL55461382.warc,webarchive,webarchive,https://ndhadeliver.natlib.govt.nz/webarchive/20200730070058/https://natlib.govt.nz/
2021-01-29 01:36:44,"nz,govt,natlib)/",http://natlib.govt.nz/,text/html,200,UAOK5GLA5X7OKWOTZI4JPXTNQK7TVKKC,-,-,0,86249495,V1-FL62209940.warc,webarchive,webarchive,https://ndhadeliver.natlib.govt.nz/webarchive/20210129013644/http://natlib.govt.nz/
2021-01-30 06:00:56,"nz,govt,natlib)/",http://natlib.govt.nz/,text/html,503,N67J36CWSVSGPQLJCVMHS3EG7Q4S5VNW,-,-,0,155090,V1-FL62215104.warc,webarchive,webarchive,https://ndhadeliver.natlib.govt.nz/webarchive/20210130060056/http://natlib.govt.nz/
2021-12-03 01:31:57,"nz,govt,natlib)/",https://natlib.govt.nz/,text/html,200,VXIQIOHXS7HN7B6YVNWYHDUEFRQBWUGK,-,-,0,3819,V1-FL80155252.warc,webarchive,webarchive,https://ndhadeliver.natlib.govt.nz/webarchive/20211203013157/https://natlib.govt.nz/
2022-04-01 09:56:43,"nz,govt,natlib)/",http://natlib.govt.nz/,text/html,301,DFS4JFJMZDAIFJRQP3LHAYNFPKVWMMX2,-,-,0,31898460,V1-FL80790464.warc,webarchive,webarchive,https://ndhadeliver.natlib.govt.nz/webarchive/20220401095643/http://natlib.govt.nz/


>💡 <strong>Try it:</strong> Copy the oldest and most recent memento URLs (from the *access_url* column) and paste them into a web browser, to compare the visual difference between the harvests.

### URL Modifiers in Memento

Memento supports special URL modifiers that control how archived content is presented:

- **mp_** modifier: Shows "main page" content only
- **id_** modifier: Returns the original harvested version without rewriting
- **if_** modifier: Shows the page with web archive headers (default for NLNZ web archive)

For more details on URL rewriting options, see the [PyWB documentation](https://pywb.readthedocs.io/en/latest/manual/rewriter.html?highlight=id_#url-rewriting).

## 2. Querying Web Archives with the CDX API

![image](https://ndha-public-data-ap-southeast-2.s3.ap-southeast-2.amazonaws.com/iPRES-2025/resources/outback-cdx.png)

The CDX (Capture inDeX) API provides a more direct way to query web archive metadata. It allows for more specific filtering and returns structured data about archived captures.

> ***Note***
>
> *In the NLNZ web archive, CDX API queries are routed via the PyWB viewer. This means some native CDX query parameters (like output format) are not supported.*

In [ ]:
# Query the CDX index for the National Library website

# New Zealand Web Archive
webpage = "www.natlib.govt.nz"

# Norwegian Web Archive
# webpage = "www.met.no"

# Australian Web Archive
# webpage = "cobb.qm.qld.gov.au/"

# Arquivo.pt
# webpage = "https://www.abola.pt"

df_captures = want.query_cdx_index(webpage)
df_captures

,urlkey,url,mime,status,digest,redirect,robotflags,length,offset,filename,source,source-coll,access_url
timestamp,,,,,,,,,,,,,
2004-07-11 21:32:25,"nz,govt,natlib)/",http://www.natlib.govt.nz/,text/html,200,JV66FPIIX6IJTB42TNHMQDEU5Z3LFBCK,-,-,0,976,V1-FL1645590.arc,webarchive,webarchive,https://ndhadeliver.natlib.govt.nz/webarchive/20040711213225/http://www.natlib.govt.nz/
2006-07-04 03:31:35,"nz,govt,natlib)/",http://www.natlib.govt.nz/,text/html,200,JKXIM5NTOXWFNC5UOIAN37AGPV2KL73O,-,-,0,976,V1-FL1645520.arc,webarchive,webarchive,https://ndhadeliver.natlib.govt.nz/webarchive/20060704033135/http://www.natlib.govt.nz/
2007-03-22 04:15:46,"nz,govt,natlib)/",http://www.natlib.govt.nz/,text/html,200,CU2KIAIJGUZD4IOV43D7LE2J5TVMUJYR,-,-,0,19799,V1-FL481509.arc,webarchive,webarchive,https://ndhadeliver.natlib.govt.nz/webarchive/20070322041546/http://www.natlib.govt.nz/
2008-02-25 06:02:38,"nz,govt,natlib)/",http://www.natlib.govt.nz/,text/html,200,2IIVSKCHCNVKN6Z273YKZBEW6QVMYXKK,-,-,0,2717767,V1-FL538322.arc,webarchive,webarchive,https://ndhadeliver.natlib.govt.nz/webarchive/20080225060238/http://www.natlib.govt.nz/
2008-10-19 22:53:43,"nz,govt,natlib)/",http://www.natlib.govt.nz/,text/html,200,6TCIF3SQHDMFZWZ2YTJ5AFNTSYUPYZX7,-,-,0,48523900,V1-FL617362.arc,webarchive,webarchive,https://ndhadeliver.natlib.govt.nz/webarchive/20081019225343/http://www.natlib.govt.nz/
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-11-26 08:35:22,"nz,govt,natlib)/",https://natlib.govt.nz/,text/html,200,7VHUCJVAF75V2YXGQHXMXNSHWW4VMPZG,-,-,-,3859,V1-FL96382746.warc,webarchive,webarchive,https://ndhadeliver.natlib.govt.nz/webarchive/20251126083522/https://natlib.govt.nz/
2025-11-26 08:44:14,"nz,govt,natlib)/",http://natlib.govt.nz/,text/html,301,DFS4JFJMZDAIFJRQP3LHAYNFPKVWMMX2,-,-,-,61550319,V1-FL96382746.warc,webarchive,webarchive,https://ndhadeliver.natlib.govt.nz/webarchive/20251126084414/http://natlib.govt.nz/
2026-01-29 06:00:48,"nz,govt,natlib)/",http://natlib.govt.nz/,text/html,301,DFS4JFJMZDAIFJRQP3LHAYNFPKVWMMX2,-,-,-,238200,V1-FL97054608.warc,webarchive,webarchive,https://ndhadeliver.natlib.govt.nz/webarchive/20260129060048/http://natlib.govt.nz/


### CDX vs TimeMap

The CDX query results are similar to the TimeMap. ***Note***, our toolkit is appending an `access_url` column that contains the actual URL for accessing each webpage snapshot. This makes it easier to view or analyze specific captures.

### Advanced CDX Queries

The CDX API allows for more specific queries, such as filtering by MIME type or using prefix matching. This is particularly useful for finding non-HTML content like images or documents.

In [ ]:
# Query for PDF files in a specific section of a website
# Note: Due to architecture limitations, we need to specify at least the first-level path segment (e.g. /assets/)
webpage = "covid19.govt.nz/assets/"

# Filter for PDF files using the MIME type filter
df_captures = want.query_cdx_index(webpage, filter="mimetype:application/pdf", matchType="prefix")

# Extract original filenames from the URLs
df_captures["original_file_name"] = df_captures["urlkey"].str.split("/").str[-1]
df_captures

>💡 <strong>Hands-On Exercise:</strong> Querying for Image Files
>
> Try completeing the code below and querying the CDX index for PNG image files from the same website section. What other MIME types can you find?
>
> See the *Hands-On Exercise Solutions* section at the end for a working solution

In [ ]:
# Exercise: Query for PNG files
webpage = "covid19.govt.nz/assets/"

# 1. Query the CDX index, fitlering on the MIME type
# 2. Extract original filenames from the URLs
# 3. Output the results

## Conclusion and Next Steps

This notebook has introduced the fundamental methods for accessing and working with open web archives:

1. **Memento Protocol** - For standardized access to archived web content
2. **TimeMaps** - For listing all mementos of an archived resource
3. **CDX API** - For querying and filtering archive metadata

### What's Next?

In the following notebooks, we'll build on these foundations to:

- Explore and analyze web archive data in more depth
- Track changes in websites over time
- Extract and analyze textual content at scale
- Build advanced applications using web archive data

These techniques provide powerful tools for researchers, historians, and data scientists working with web archives.

---

## Hands-On Exercise Solutions

#### Querying for Image Files

```
# Exercise: Query for PNG files
webpage = "covid19.govt.nz/assets/"

df_captures = want.query_cdx_index(webpage, filter="mimetype:image/png", matchType="prefix")
df_captures["original_file_name"] = df_captures["urlkey"].str.split("/").str[-1]
df_captures
```